# HGND — head-to-head model comparison

Small interactive driver for comparing models on the same subset of shards.
Uses the model registry (`HGNDRecoGNN.models`) so any registered model plugs in with a name change.
For heavyweight runs use `scripts/train.py` + SLURM instead — this notebook is for iterating on architecture choices.

In [ ]:
import os, sys, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline
import torch

PACKAGE_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PACKAGE_ROOT not in sys.path:
    sys.path.insert(0, PACKAGE_ROOT)

from HGNDRecoGNN import device as device_mod, models as model_registry
from HGNDRecoGNN.data.graph_dataset import HGNDGraphDataset
from HGNDRecoGNN.training import TrainConfig, fit, LossWeights, HeteroClusterWeights
from HGNDRecoGNN.training.train import build_loaders

print('registered models:', model_registry.available())

## 1. Small dataset slice

In [ ]:
DATASET_NAME = 'defaultSpot'
DATASET_ROOT = os.path.join(os.getcwd(), 'cache', f'ndet_dataset_smash_{DATASET_NAME}')
NUM_SHARDS   = 4       # ~4k graphs → enough for a quick head-to-head
BATCH        = 128
EPOCHS       = 3       # keep short; this is a comparison harness
SEED         = 42

dataset = HGNDGraphDataset(root=DATASET_ROOT, hits_csv_dir=DATASET_ROOT,
                           num_shards=NUM_SHARDS)
dataset.preload()
print(f'{len(dataset)} graphs loaded from {NUM_SHARDS} shards')

## 2. Loop over models

In [ ]:
MODELS_TO_COMPARE = [
    'net_default',
    'hetero_sage',
    'hetero_gat',
    'hetero_hgt',
    'thesis_sage',
    'thesis_dynedge',
    'spectral_dynedge',
]

results = {}
for name in MODELS_TO_COMPARE:
    print(f'\n=== {name} ===')
    torch.manual_seed(SEED)
    
    model, spec = model_registry.get(name, dataset)
    plan = device_mod.plan_for(model, 'auto',
                               cpu_pinned=spec.default_cpu_pinned or None)
    device_mod.to_device(model, plan)
    print(f'  {device_mod.summarize(plan)}  params={sum(p.numel() for p in model.parameters()):,}')
    
    cfg = TrainConfig(
        batch_size=BATCH, seed=SEED, epochs=EPOCHS,
        arch_name=name, loss_weights=spec.default_loss_weights,
        checkpoint_dir=os.path.join(os.getcwd(), 'checkpoints', 'compare'),
        checkpoint_name=f'{name}.pt', verbose=False,
    )
    train_loader, test_loader = build_loaders(dataset, cfg)
    
    t0 = time.time()
    result = fit(model, train_loader, test_loader, cfg, plan=plan,
                 forward_fn=spec.forward_fn, loss_fn=spec.loss_fn)
    result['elapsed'] = time.time() - t0
    result['params'] = sum(p.numel() for p in model.parameters())
    results[name] = result
    print(f'  best val {result["best_loss"]:.4f} @ epoch {result["best_epoch"]} '
          f'({result["elapsed"]:.0f}s)')

## 3. Compare

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        'model': name,
        'params (M)': r['params'] / 1e6,
        'best epoch': r['best_epoch'],
        'best val loss': r['best_loss'],
        'final train': r['train_loss'][-1],
        'elapsed (s)': r['elapsed'],
    })
df = pd.DataFrame(rows).sort_values('best val loss')
df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, r in results.items():
    ax[0].plot(r['train_loss'], label=name)
    ax[1].plot(r['val_loss'],   label=name)
ax[0].set_title('Train loss');  ax[0].set_xlabel('epoch');  ax[0].grid(True); ax[0].legend(fontsize=8)
ax[1].set_title('Val loss');    ax[1].set_xlabel('epoch');  ax[1].grid(True); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()